### Setup

In [1]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from app.config import get_settings

settings = get_settings()
os.environ["HF_TOKEN"] = settings.hf_token

print({k: ("set" if v else "MISSING") for k, v in settings.model_dump().items()})

{'app_env': 'set', 'database_url': 'set', 'groq_api_key': 'set', 'groq_model': 'set', 'hf_token': 'set', 'embedding_model': 'set', 'fernet_key': 'set', 'llm_cache_dir': 'set', 'mem0_api_key': 'set', 'demo_database_url': 'set'}


In [2]:
from sqlalchemy import create_engine, text

engine = create_engine(settings.database_url, pool_pre_ping=True)


def sql(query: str, **params) -> pd.DataFrame:
    """Run a query on the app database and return a DataFrame."""
    with engine.connect() as conn:
        res = conn.execute(text(query), params)
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


display(sql("select version()"))
display(sql("select extname, extversion from pg_extension where extname = 'vector'"))

,version
0,"PostgreSQL 17.11 on aarch64-unknown-linux-gnu,..."


,extname,extversion
0,vector,0.8.2


In [3]:
from groq import APIConnectionError, Groq, RateLimitError

client = Groq(api_key=settings.groq_api_key)

In [4]:
CACHE_DIR = Path(settings.llm_cache_dir)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def llm(messages, *, model=None, temperature=0.0, json_mode=False, use_cache=True, **kwargs) -> str:
    """Single entry point for all LLM calls. Cached on disk so reruns are free and reproducible."""
    model = model or settings.groq_model
    payload = {"model": model, "messages": messages, "temperature": temperature,
               "json_mode": json_mode, **kwargs}
    key = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    path = CACHE_DIR / f"{key}.json"
    if use_cache and path.exists():
        return json.loads(path.read_text(encoding="utf-8"))["text"]

    params = dict(model=model, messages=messages, temperature=temperature, **kwargs)
    if json_mode:
        params["response_format"] = {"type": "json_object"}

    for attempt in range(4):
        try:
            resp = client.chat.completions.create(**params)
            break
        except (RateLimitError, APIConnectionError):
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

    text_out = resp.choices[0].message.content
    path.write_text(json.dumps({"text": text_out}), encoding="utf-8")
    return text_out


msgs = [{"role": "user", "content": "Reply with exactly one word: ready"}]
for attempt in (1, 2):
    t0 = time.perf_counter()
    print(attempt, llm(msgs), f"{time.perf_counter() - t0:.2f}s")

1 ready 0.00s
2 ready 0.00s


In [5]:
import httpx

get_settings.cache_clear()
settings = get_settings()

HF_EMBED_URL = (
    "https://router.huggingface.co/hf-inference/models/"
    f"{settings.embedding_model}/pipeline/feature-extraction"
)

EMB_DIR = Path(settings.llm_cache_dir).parent / "emb"
EMB_DIR.mkdir(parents=True, exist_ok=True)


def _emb_path(text: str) -> Path:
    key = hashlib.sha256(f"{settings.embedding_model}\n{text}".encode()).hexdigest()
    return EMB_DIR / f"{key}.json"


def _hf_embed_batch(texts: list[str], max_retries: int = 4) -> list[list[float]]:
    headers = {"Authorization": f"Bearer {settings.hf_token}"}
    with httpx.Client(timeout=30) as http:
        for attempt in range(max_retries):
            resp = http.post(HF_EMBED_URL, headers=headers, json={"inputs": texts})
            if resp.status_code == 200:
                data = resp.json()
                ok = isinstance(data, list) and data and isinstance(data[0], list) \
                    and isinstance(data[0][0], (int, float))
                if not ok:
                    raise ValueError(f"Unexpected embedding shape from HF: {str(data)[:200]}")
                return data
            if resp.status_code in (429, 503):  # rate limited, or model cold-starting
                wait = 2 ** (attempt + 1)
                if resp.status_code == 503:
                    try:
                        wait = min(float(resp.json().get("estimated_time", wait)), 20)
                    except Exception:  # noqa: S110
                        pass
                print(f"HF returned {resp.status_code}, retrying in {wait:.0f}s")
                time.sleep(wait)
                continue
            raise RuntimeError(f"HF embedding error {resp.status_code}: {resp.text[:300]}")
    raise RuntimeError(f"HF embedding API failed after {max_retries} retries")


def embed(texts: list[str], *, use_cache: bool = True, batch_size: int = 32) -> list[list[float]]:
    """Embed texts, caching each one on disk so reruns are free and reproducible."""
    out = [None] * len(texts)
    missing = []
    for i, t in enumerate(texts):
        p = _emb_path(t)
        if use_cache and p.exists():
            out[i] = json.loads(p.read_text(encoding="utf-8"))
        else:
            missing.append(i)
    for s in range(0, len(missing), batch_size):
        idx = missing[s:s + batch_size]
        vecs = _hf_embed_batch([texts[i] for i in idx])
        for i, v in zip(idx, vecs):
            out[i] = v
            _emb_path(texts[i]).write_text(json.dumps(v), encoding="utf-8")
    return out


def embed_query(text: str) -> list[float]:
    return embed([text])[0]


def cos(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


vecs = embed(["customer orders", "product price", "customer purchases"])
print("vectors:", len(vecs), "| dimension:", len(vecs[0]))
print("orders vs purchases:", round(cos(vecs[0], vecs[2]), 3))
print("orders vs price:    ", round(cos(vecs[0], vecs[1]), 3))

vectors: 3 | dimension: 384
orders vs purchases: 0.738
orders vs price:     0.247


### Demo data

In [6]:
tables = ["customers", "categories", "products", "orders", "order_items", "payments", "reviews"]
display(pd.DataFrame({
    "table": tables,
    "rows": [int(sql(f"select count(*) as c from demo.{t}")["c"][0]) for t in tables],
}))

,table,rows
0,customers,500
1,categories,15
2,products,120
3,orders,3000
4,order_items,6252
5,payments,2958
6,reviews,1184


In [7]:
demo_engine = create_engine(settings.demo_database_url, pool_pre_ping=True)


def demo_sql(query: str, **params) -> pd.DataFrame:
    """Run a query on the demo data through the read-only role."""
    with demo_engine.connect() as conn:
        res = conn.execute(text(query), params)
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


display(demo_sql("""
    select current_user,
           current_setting('search_path') as search_path,
           current_setting('default_transaction_read_only') as read_only,
           current_setting('statement_timeout') as statement_timeout
"""))

,current_user,search_path,read_only,statement_timeout
0,askdata_ro,demo,on,15s


In [8]:
for stmt in [
    "insert into customers (customer_id, full_name, email, signup_date, cust_stat_cd) "
    "values (999999, 'x', 'x@example.com', '2026-01-01', 'A')",
    "update orders set ord_sts = 'C'",
    "drop table customers",
]:
    try:
        with demo_engine.begin() as conn:
            conn.execute(text(stmt))
        print("!! WAS ALLOWED:", stmt[:50])
    except Exception as e:
        print("refused:", str(getattr(e, "orig", e)).splitlines()[0])

refused: cannot execute INSERT in a read-only transaction
refused: cannot execute UPDATE in a read-only transaction
refused: cannot execute DROP TABLE in a read-only transaction


In [9]:
display(demo_sql("select ord_sts, count(*) as orders from orders group by ord_sts order by orders desc"))

display(demo_sql("""
    select c.customer_id, c.full_name, round(sum(o.total_amount), 2) as revenue
    from customers c
    join orders o on o.customer_id = c.customer_id
    where o.ord_sts = 'D'
    group by c.customer_id, c.full_name
    order by revenue desc
    limit 5
"""))

display(demo_sql("""
    select date_trunc('month', order_date)::date as month, round(sum(total_amount), 2) as revenue
    from orders
    where ord_sts = 'D'
    group by 1
    order by 1
"""))

,ord_sts,orders
0,D,2312
1,C,372
2,R,239
3,P,42
4,S,35


,customer_id,full_name,revenue
0,461,Neel Mani,18658077.15
1,389,Neel Sharaf,5785237.00
2,426,Faqid Wason,3326826.55
3,274,Urishilla Saran,2427404.05
4,244,Samar Bali,1975657.90


,month,revenue
0,2025-01-01,3029376.25
1,2025-02-01,1754115.65
2,2025-03-01,2122381.35
3,2025-04-01,1959318.15
4,2025-05-01,1964747.15
5,2025-06-01,1497545.00
6,2025-07-01,2262276.50
7,2025-08-01,3127553.55
8,2025-09-01,3142631.35
9,2025-10-01,3221696.75
